# NBME - DeBERTa-v3-base Inference (Colab)

**前置條件**：必須先跑完 `train-deberta-v3-base-colab.ipynb`，產出：
- `oof_df.pkl`：全部 fold 的驗證集預測（用來調 threshold）
- `microsoft-deberta-v3-base_fold{0~4}_best.pth`：5 個 fold 的最佳模型權重
- `config.pth`：模型架構設定檔
- `tokenizer/`：tokenizer 存檔

## 整體流程
```
Step 1: 讀 oof_df.pkl → threshold tuning（找讓 F1 最高的 threshold）
Step 2: 讀 test_preprocessed.pkl → 5 fold 各跑一次 inference
Step 3: 平均 5 個 fold 的 char-level 機率（ensemble）
Step 4: 用最佳 threshold decode → 輸出 submission.csv
```

# Colab Setup

In [ ]:
# transformers：模型與 tokenizer 主要套件
# sentencepiece：DeBERTa-v3 tokenizer 的必要相依套件
!pip install -q transformers sentencepiece

from google.colab import drive
drive.mount('/content/drive')

# Directory Settings

In [ ]:
import os

# ========== 請修改成你的路徑（和訓練 notebook 一致）==========
BASE_DIR      = '/content/drive/MyDrive/NBME_Score-Clinical-Patient-Notes'
# 訓練 notebook 的輸出目錄（存放模型權重、oof_df、config、tokenizer）
MODEL_DIR     = BASE_DIR + '/outputs/deberta-v3-base/'
# test_preprocessed.pkl：preprocess_data.ipynb 產出的測試集
TEST_PKL_PATH = BASE_DIR + '/data/nbme-score-clinical-patient-notes/processed/test_preprocessed.pkl'
# submission.csv 的輸出位置
OUTPUT_DIR    = BASE_DIR + '/outputs/deberta-v3-base/'
# ============================================================

print(f'MODEL_DIR:     {MODEL_DIR}')
print(f'TEST_PKL_PATH: {TEST_PKL_PATH}')

# CFG

inference 用的 CFG 必須和**訓練時保持一致**，尤其是：
- `model`：模型名稱（決定架構）
- `fc_dropout`：雖然 inference 時 Dropout 會自動關閉，但架構定義要一致
- `max_len`：這裡設為 None，之後從 oof_df 自動推斷，確保和訓練時相同

In [ ]:
class CFG:
    model       = 'microsoft/deberta-v3-base'  # 必須和訓練時一致
    config_path = MODEL_DIR + 'config.pth'     # 訓練時用 torch.save(model.config) 存下的
    batch_size  = 32      # inference 不做 backward，記憶體壓力小，可以用更大的 batch
    fc_dropout  = 0.2     # 必須和訓練時一致（inference 時 model.eval() 會自動關掉 dropout）
    max_len     = None    # 從 oof_df 自動推斷，確保和訓練時使用的 max_len 相同
    seed        = 42
    num_workers = 2
    n_fold      = 5
    trn_fold    = [0, 1, 2, 3, 4]  # 要做 ensemble 的 fold，全部跑才能充分利用所有模型

# Library

In [ ]:
import os
import gc
import re
import random
import itertools
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

from tqdm.auto import tqdm
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

import tokenizers
import transformers
print(f'tokenizers.__version__: {tokenizers.__version__}')
print(f'transformers.__version__: {transformers.__version__}')

from transformers import AutoModel, AutoConfig
from transformers import DebertaV2TokenizerFast  # deberta-v3 專用 fast tokenizer

%env TOKENIZERS_PARALLELISM=true

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

# Utils

In [ ]:
def seed_everything(seed=42):
    """固定所有隨機種子，確保結果可重現"""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(seed=CFG.seed)

# Scoring Functions

和訓練 notebook 完全相同，用於 threshold tuning 時計算 OOF 的 F1 分數。

In [ ]:
def micro_f1(preds, truths):
    """把所有樣本的預測和真實拼起來，計算整體 binary F1"""
    preds  = np.concatenate(preds)
    truths = np.concatenate(truths)
    return f1_score(truths, preds)


def spans_to_binary(spans, length=None):
    """把 span list 轉成字元層級的 0/1 陣列，方便計算 F1"""
    length = np.max(spans) if length is None else length
    binary = np.zeros(length)
    for start, end in spans:
        binary[start:end] = 1
    return binary


def span_micro_f1(preds, truths):
    """競賽官方指標：character-level micro F1"""
    bin_preds  = []
    bin_truths = []
    for pred, truth in zip(preds, truths):
        # 預測和真實都是空的，不影響分數，跳過
        if not len(pred) and not len(truth):
            continue
        length = max(
            np.max(pred)  if len(pred)  else 0,
            np.max(truth) if len(truth) else 0
        )
        bin_preds.append(spans_to_binary(pred, length))
        bin_truths.append(spans_to_binary(truth, length))
    return micro_f1(bin_preds, bin_truths)


def create_labels_for_scoring(df):
    """
    從 oof_df 的 char_spans 欄位建立 ground truth
    只有訓練集的 oof_df 才有 char_spans，測試集沒有
    """
    truths = []
    for char_spans in df['char_spans'].values:
        truth = [[start, end] for start, end in char_spans]
        truths.append(truth)
    return truths


def get_char_probs(texts, predictions, tokenizer):
    """
    把 token-level 機率對應回 character-level 機率

    每個 token 有 offset_mapping 紀錄它對應到原文的哪個字元範圍，
    把那個範圍內的所有字元都填上這個 token 的機率。
    """
    results = [np.zeros(len(t)) for t in texts]
    for i, (text, prediction) in enumerate(zip(texts, predictions)):
        encoded = tokenizer(text, add_special_tokens=True, return_offsets_mapping=True)
        for offset_mapping, pred in zip(encoded['offset_mapping'], prediction):
            start, end = offset_mapping
            results[i][start:end] = pred
    return results


def get_results(char_probs, th=0.5):
    """
    character 機率 → span 字串

    1. 找出機率 >= threshold 的字元位置
    2. 把連續位置合併成一個 span
    3. 多個 span 用分號串接，例如 '70 91;176 183'
    4. 無預測則回傳空字串 ''
    """
    results = []
    for char_prob in char_probs:
        result = np.where(char_prob >= th)[0] + 1
        result = [list(g) for _, g in itertools.groupby(
            result, key=lambda n, c=itertools.count(): n - next(c))]
        result = [f'{min(r)} {max(r)}' for r in result]
        results.append(';'.join(result))
    return results


def get_predictions(results):
    """span 字串 → list of [[start, end], ...]，供 F1 計算使用"""
    predictions = []
    for result in results:
        prediction = []
        if result != '':
            for loc in [s.split() for s in result.split(';')]:
                start, end = int(loc[0]), int(loc[1])
                prediction.append([start, end])
        predictions.append(prediction)
    return predictions


def get_score(y_true, y_pred):
    return span_micro_f1(y_true, y_pred)

# Tokenizer

從訓練時存下的 `tokenizer/` 資料夾載入，確保使用完全相同的 tokenizer 設定。

In [ ]:
# 從訓練時存下的 tokenizer 資料夾載入
# 這樣可以確保 tokenization 結果和訓練時完全一致
tokenizer = DebertaV2TokenizerFast.from_pretrained(MODEL_DIR + 'tokenizer/')
CFG.tokenizer = tokenizer
print('Tokenizer loaded.')

# Step 1：Threshold Tuning

**為什麼要調 threshold？**

模型輸出的是每個字元「是答案」的機率（0~1），
需要設一個切分點（threshold）來決定哪些字元要被納入預測答案。

- threshold 太高 → 預測太保守，FN 多，recall 低
- threshold 太低 → 預測太積極，FP 多，precision 低

**為什麼用 OOF 來調？**

`oof_df` 是每個 fold 驗證集的預測，每筆資料都只被當過一次驗證集（沒有洩漏），
等同於模型在未見資料上的表現，用它來選 threshold 不會 overfitting。

In [ ]:
# 讀入訓練 notebook 產出的 oof_df
oof = pd.read_pickle(MODEL_DIR + 'oof_df.pkl')
print(f'oof.shape: {oof.shape}')

# oof_df 的欄位 0, 1, 2, ..., max_len-1 是每個 token 的預測機率
# 用欄位名稱是整數的欄位數量來推斷訓練時的 max_len
pred_cols   = [c for c in oof.columns if isinstance(c, int)]
CFG.max_len = len(pred_cols)
print(f'max_len from oof_df: {CFG.max_len}')

In [ ]:
# ground truth：從 char_spans 建立
truths = create_labels_for_scoring(oof)

# oof 的 token-level 預測機率 → char-level 機率
char_probs = get_char_probs(
    oof['pn_history'].values,
    oof[[i for i in range(CFG.max_len)]].values,  # shape: [n_train, max_len]
    CFG.tokenizer
)

# 試驗不同的 threshold，找出讓 F1 最高的值
best_th    = 0.5
best_score = 0.

print('Threshold tuning:')
for th in np.arange(0.45, 0.56, 0.01):
    th      = np.round(th, 2)
    results = get_results(char_probs, th=th)
    preds   = get_predictions(results)
    score   = get_score(preds, truths)
    if best_score < score:
        best_th    = th
        best_score = score
    print(f'  th={th:.2f}  score={score:.5f}')

print(f'\nbest_th={best_th}  best_score={best_score:.5f}')

# Step 2：Test Data Loading

直接讀 `test_preprocessed.pkl`，已合併三表，不需要再做 merge。

**注意**：feature_text 前處理必須和訓練時完全一樣，
否則 tokenization 結果會不同，模型看到的輸入格式就不一致。

In [ ]:
def process_feature_text(text):
    """
    和訓練時完全相同的 feature_text 前處理
    原始：'Chest-pressure' → 處理後：'Chest pressure'
    """
    text = re.sub('I-year', '1-year', text)  # 修正 OCR 錯誤
    text = re.sub('-OR-', ' or ', text)       # -OR- 換成自然語言
    text = re.sub('-', ' ', text)             # 其餘 dash 換空白
    return text


# 直接讀 pkl，不需要 merge
test = pd.read_pickle(TEST_PKL_PATH)
test['feature_text'] = test['feature_text'].apply(process_feature_text)

print(f'test.shape: {test.shape}')
print(f'columns: {test.columns.tolist()}')
display(test.head())

# Test Dataset

測試集不需要 label，只需要把 pn_history + feature_text tokenize 成模型輸入格式。

tokenization 方式和訓練時的 `prepare_input` 完全相同。

In [ ]:
def prepare_input(cfg, text, feature_text):
    """和訓練時完全相同的 tokenization，格式不一致模型預測會出錯"""
    inputs = cfg.tokenizer(
        text, feature_text,         # pn_history 在前，feature_text 在後
        add_special_tokens=True,
        max_length=CFG.max_len,     # 和訓練時相同的 max_len
        padding='max_length',
        return_offsets_mapping=False,
    )
    for k, v in inputs.items():
        inputs[k] = torch.tensor(v, dtype=torch.long)
    return inputs


class TestDataset(Dataset):
    """測試集 Dataset，__getitem__ 只回傳 inputs，不回傳 label"""
    def __init__(self, cfg, df):
        self.cfg           = cfg
        self.feature_texts = df['feature_text'].values
        self.pn_historys   = df['pn_history'].values

    def __len__(self):
        return len(self.feature_texts)

    def __getitem__(self, item):
        return prepare_input(self.cfg, self.pn_historys[item], self.feature_texts[item])

# Model

架構必須和訓練時**完全一致**。

inference 時用 `pretrained=False`，因為權重從 `.pth` 檔載入，
不需要也不應該再從 HuggingFace 下載預訓練權重（那是訓練前的初始權重）。

用 `config.pth`（訓練時存下的）而非重新從 HuggingFace 抓 config，
確保架構完全一致，不會因版本差異導致 state_dict 載入失敗。

In [ ]:
class CustomModel(nn.Module):
    def __init__(self, cfg, config_path=None, pretrained=False):
        super().__init__()
        self.cfg = cfg
        # 優先使用訓練時存下的 config，避免版本差異
        if config_path is None:
            self.config = AutoConfig.from_pretrained(cfg.model, output_hidden_states=True)
        else:
            self.config = torch.load(config_path)
        # inference 時 pretrained=False：只建立架構，等一下載入 .pth 權重
        if pretrained:
            self.model = AutoModel.from_pretrained(cfg.model, config=self.config)
        else:
            self.model = AutoModel.from_config(self.config)
        self.fc_dropout = nn.Dropout(cfg.fc_dropout)  # inference 時 model.eval() 會自動關閉
        self.fc = nn.Linear(self.config.hidden_size, 1)

    def feature(self, inputs):
        outputs = self.model(**inputs)
        return outputs[0]  # last hidden states: [batch, seq_len, hidden_size]

    def forward(self, inputs):
        feature = self.feature(inputs)
        # .float()：DeBERTa-v3 backbone 可能輸出 FP16，強制轉成 FP32 才能和 fc 層相容
        return self.fc(self.fc_dropout(feature.float()))

# Step 3：Inference + Ensemble

**為什麼要 ensemble？**

每個 fold 的模型看到的訓練資料不同，預測時各有偏差，
把多個模型的機率取平均可以讓誤差互相抵消，通常可以提升分數。

**為什麼在 char-level 機率階段平均，而不是 span 投票？**

在機率階段平均保留了更細緻的資訊（每個字元的信心程度），
比各模型先 decode 成 span 再投票更穩定。

```
fold0: char_probs = [0.1, 0.9, 0.8, 0.2, ...]
fold1: char_probs = [0.2, 0.7, 0.9, 0.1, ...]  →  平均  →  threshold  →  span
fold2: char_probs = [0.1, 0.8, 0.85, 0.3, ...]
```

In [ ]:
def inference_fn(test_loader, model, device):
    """
    對 test_loader 跑 inference
    回傳 sigmoid 後的 token-level 機率，shape: [n_test, max_len, 1]
    """
    preds = []
    model.eval()    # 關閉 Dropout、BatchNorm 等訓練模式
    model.to(device)

    for inputs in tqdm(test_loader, total=len(test_loader)):
        for k, v in inputs.items():
            inputs[k] = v.to(device)
        with torch.no_grad():  # 不計算梯度，節省記憶體和時間
            y_preds = model(inputs)
        # sigmoid：把 logit 轉成機率 [0, 1]
        preds.append(y_preds.sigmoid().cpu().numpy())

    return np.concatenate(preds)  # [n_test, max_len, 1]

In [ ]:
test_dataset = TestDataset(CFG, test)
test_loader  = DataLoader(
    test_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,       # inference 時不打亂順序，確保結果能對應回原始資料
    num_workers=CFG.num_workers,
    pin_memory=True,
    drop_last=False,     # inference 時不丟棄最後一個不完整的 batch
)

all_char_probs = []  # 收集每個 fold 的 char-level 機率，最後取平均

for fold in CFG.trn_fold:
    print(f'\n========== fold: {fold} inference ==========')

    # 建立模型架構（用 config.pth，不從 HuggingFace 下載預訓練權重）
    model = CustomModel(CFG, config_path=CFG.config_path, pretrained=False)

    # 載入這個 fold 訓練出來的最佳權重
    weight_path = MODEL_DIR + f"{CFG.model.replace('/', '-')}_fold{fold}_best.pth"
    state = torch.load(weight_path, map_location='cpu')
    model.load_state_dict(state['model'])
    print(f'Loaded weights: {weight_path}')

    # 執行 inference，取得 token-level 機率
    prediction = inference_fn(test_loader, model, device)
    prediction = prediction.reshape((len(test), CFG.max_len))  # [n_test, max_len]

    # token 機率 → char 機率（把 token 機率展開到對應的字元範圍）
    char_probs = get_char_probs(test['pn_history'].values, prediction, CFG.tokenizer)
    all_char_probs.append(char_probs)

    # 釋放 GPU 記憶體，避免跑到第 2、3 個 fold 時 OOM
    del model, state, prediction, char_probs
    gc.collect()
    torch.cuda.empty_cache()

# 5 個 fold 的 char-level 機率取平均（ensemble）
# all_char_probs: list of [n_test, n_chars]，每個 fold 的 char 長度不同所以是 list
final_char_probs = np.mean(all_char_probs, axis=0)
print(f'\nEnsemble done. {len(CFG.trn_fold)} folds averaged.')

# Step 4：Submission

用 Step 1 找到的最佳 threshold，把 char-level 機率轉成 span 字串，輸出 submission.csv。

submission 格式：
```
id,location
00016_000,203 217
00016_001,
00016_002,70 91;176 183
```
- 有答案：`start end`，多段用 `;` 分隔
- 無答案：空字串

In [ ]:
# 用最佳 threshold 把 char 機率轉成 span 字串
results = get_results(final_char_probs, th=best_th)

# 組成 submission DataFrame
submission = test[['id']].copy()
submission['location'] = results

print(f'使用的 threshold: {best_th}')
print(f'有預測答案的筆數: {(submission["location"] != "").sum()} / {len(submission)}')
display(submission.head(10))

# 存成 csv
sub_path = OUTPUT_DIR + 'submission.csv'
submission[['id', 'location']].to_csv(sub_path, index=False)
print(f'\nSaved: {sub_path}')

# Quick Check

確認輸出格式符合競賽要求。

In [ ]:
# 重新讀入確認格式正確
sub = pd.read_csv(sub_path)
print(f'submission shape: {sub.shape}')
print(f'欄位: {sub.columns.tolist()}')
print(f'空預測（無答案）筆數: {(sub["location"].fillna("") == "").sum()}')
display(sub)